<a href="https://colab.research.google.com/github/KJ2220/delivery-parking-map/blob/main/delivery-parking-map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import requests
import pandas as pd
import time

query = """
[out:json][timeout:60];
nwr["amenity"="parking"](52.10,-0.52,52.18,-0.40);
out center tags;
"""

servers = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]
headers = {"User-Agent": "parking-map-learning-project/1.0"}

data = None
for url in servers:
    try:
        r = requests.post(url, data={"data": query}, headers=headers, timeout=90)
        print(url, "->", r.status_code)
        if r.status_code == 200:
            data = r.json()
            break
    except Exception as e:
        print(url, "failed:", e)
    time.sleep(2)

if data is None:
    raise SystemExit("All servers failed. Wait a few minutes and run again.")

rows = []
for el in data["elements"]:
    lat = el.get("lat") or el.get("center", {}).get("lat")
    lon = el.get("lon") or el.get("center", {}).get("lon")
    tags = el.get("tags", {})
    rows.append({
        "name": tags.get("name", "Unnamed parking"),
        "type": tags.get("parking", "unknown"),
        "fee": tags.get("fee", "unknown"),
        "access": tags.get("access", "unknown"),
        "lat": lat,
        "lon": lon,
    })

df = pd.DataFrame(rows).dropna(subset=["lat", "lon"])
print(len(df), "parking places found")
df.head(10)

https://overpass-api.de/api/interpreter -> 200
328 parking places found


,name,type,fee,access,lat,lon
0,Unnamed parking,layby,unknown,unknown,52.158351,-0.503108
1,Unnamed parking,layby,unknown,unknown,52.157709,-0.501552
2,Unnamed parking,unknown,unknown,unknown,52.113786,-0.481404
3,Unnamed parking,unknown,unknown,unknown,52.110261,-0.510537
4,Unnamed parking,unknown,unknown,unknown,52.109251,-0.511829
5,River Street Car Park,multi-storey,yes,yes,52.135414,-0.470660
6,Unnamed parking,unknown,unknown,unknown,52.160684,-0.487891
7,Unnamed parking,surface,unknown,unknown,52.120266,-0.518127
8,Biddenham Upper School car park,surface,no,unknown,52.137771,-0.496234
9,Harpur Centre Car Park,unknown,yes,unknown,52.135589,-0.469324


In [6]:
import folium

m = folium.Map(location=[52.136, -0.467], zoom_start=14)

for _, row in df.iterrows():
    color = "green" if row["fee"] == "no" else "red" if row["fee"] == "yes" else "gray"
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=5,
        color=color,
        fill=True,
        popup=f"{row['name']}<br>Fee: {row['fee']}<br>Type: {row['type']}",
    ).add_to(m)

m

In [7]:
print(df["fee"].value_counts())
print(df["type"].value_counts())

fee
unknown    244
no          59
yes         25
Name: count, dtype: int64
type
surface         176
unknown         120
street_side      24
multi-storey      4
layby             2
lane              2
Name: count, dtype: int64


In [8]:
m = folium.Map(location=[52.136, -0.467], zoom_start=14, tiles="CartoDB positron")

/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


In [9]:
m.save("bedford_parking.html")

from google.colab import files
files.download("bedford_parking.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import numpy as np
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="parking-map-learning-project")

def nearest_parking(place, n=10, free_only=False):
    loc = geolocator.geocode(place + ", Bedford, UK")
    if loc is None:
        print("Place not found. Try adding the street name.")
        return None, None
    lat0, lon0 = loc.latitude, loc.longitude

    d = df.copy()
    R = 6371000  # Earth radius in metres
    p1 = np.radians(lat0)
    p2 = np.radians(d["lat"])
    dphi = p2 - p1
    dlam = np.radians(d["lon"] - lon0)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlam / 2) ** 2
    d["distance_m"] = (2 * R * np.arcsin(np.sqrt(a))).round()
    d["walk_min"] = (d["distance_m"] / 80).round(1)  # about 80 m per minute

    if free_only:
        d = d[d["fee"] == "no"]

    return (lat0, lon0), d.sort_values("distance_m").head(n)

In [11]:
import folium

place = "Bedford railway station"   # change this to any restaurant or street
centre, results = nearest_parking(place, n=10)

if results is not None:
    print(results[["name", "fee", "type", "distance_m", "walk_min"]])

    m = folium.Map(location=centre, zoom_start=16, tiles="CartoDB positron")
    folium.Marker(centre, popup=place, icon=folium.Icon(color="blue")).add_to(m)

    for _, row in results.iterrows():
        color = "green" if row["fee"] == "no" else "red" if row["fee"] == "yes" else "gray"
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=7, color=color, fill=True,
            popup=f"{row['name']}<br>Fee: {row['fee']}<br>{row['distance_m']} m away",
        ).add_to(m)
    m

                                 name      fee     type  distance_m  walk_min
253  Bedford Station Premium Car Park      yes  unknown        61.0       0.8
254                   Unnamed parking  unknown  surface        61.0       0.8
54                    Unnamed parking      yes  surface        89.0       1.1
154                   Unnamed parking      yes  surface       136.0       1.7
159                   Unnamed parking  unknown  unknown       241.0       3.0
34                    Unnamed parking      yes  surface       249.0       3.1
142    Greyfriars Short Stay Car Park      yes  surface       366.0       4.6
315                   Unnamed parking  unknown  surface       380.0       4.8
192                   Unnamed parking  unknown  unknown       466.0       5.8
40                    Unnamed parking  unknown  unknown       481.0       6.0


/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


In [12]:
total = len(df)
print("Total places:", total)
print()
print("Fee info:")
print(df["fee"].value_counts())
print()
print("Free places:", (df["fee"] == "no").sum())
print("Paid places:", (df["fee"] == "yes").sum())
print("Unknown fee:", (df["fee"] == "unknown").sum())
print()
print("Percent with known fee:", round((df["fee"] != "unknown").mean() * 100), "%")

Total places: 328

Fee info:
fee
unknown    244
no          59
yes         25
Name: count, dtype: int64

Free places: 59
Paid places: 25
Unknown fee: 244

Percent with known fee: 26 %
